In [ ]:
import os
import pandas as pd
from os.path import join

In [ ]:
src = os.path.join("misinfo-panel-austria", "data", "urls")

In [ ]:
urls = pd.read_csv(join(src, "shortened_urls_r2.csv.gz"),
                     compression="gzip")
len(urls), urls.url.nunique(), urls.domain.nunique()

In [ ]:
domains_r1 = pd.read_csv(join(src, "unraveled_domains.csv.gz"), 
                     compression="gzip").drop_duplicates()

domains_r2 = pd.read_csv(join(src, "unraveled_domains_reduc.csv.gz"),
                     compression="gzip").drop_duplicates()
len(domains_r1), len(domains_r2)

(2358605, 4402391)

In [6]:
# concat domains
domains = pd.concat([domains_r1, domains_r2], axis=0)\
    .drop_duplicates()
len(domains), domains.url.nunique(), domains.domain.nunique()

(6719935, 6669148, 156579)

In [20]:
timeouts = len(domains) - len(domains["status_code"].dropna())
print("{} timeouts ({:1.2f}%)".format(\
        timeouts,
        (timeouts / len(domains["status_code"].dropna()) * 100)))

344230 timeouts (5.40%)


In [18]:
# how many missing values in unraveled_url?
nans = len(domains[domains.domain.isna()])

# whats the percentage of unsuccessful unraveled urls?
nans / len(domains) * 100

0.2600918014831989

In [7]:
# which urls in urls are not in domains?
missing = urls[~urls.url.isin(domains.url)]
len(missing), missing.url.nunique(), missing.domain.nunique()

(2015144, 1988882, 5)

In [8]:
missing.domain.value_counts()

domain
fb.me         831159
youtu.be      725434
t.co          180128
tmblr.co      172095
instagr.am    106328
Name: count, dtype: int64

In [9]:
domains.head(2)

,url,unraveled_url,status_code,domain
0,http://dlvr.it/RrvBHT,https://orf.at/stories/3199975/?utm_source=dlv...,200.0,orf.at
1,https://chng.it/ypDBXkgj,https://www.change.org/p/stoppen-sie-die-hinri...,200.0,change.org


In [10]:
missing["unraveled_url"] = missing.domain
missing.head(2)

/tmp/ipykernel_3148127/307777275.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  missing["unraveled_url"] = missing.domain


,url,domain,unraveled_url
2381400,http://fb.me/3igcB4udE,fb.me,fb.me
2381401,http://youtu.be/QypkkSY5MKU?a,youtu.be,youtu.be


In [11]:
missing.loc[missing.domain == "youtu.be", "domain"] = "youtube.com"
missing.loc[missing.domain == "t.co", "domain"] = "twitter.com"
missing.loc[missing.domain == "fb.me", "domain"] = "facebook.com"
missing.loc[missing.domain == "tmblr.co", "domain"] = "tumblr.com"
missing.loc[missing.domain == "instagr.am", "domain"] = "instagram.com"
missing.domain.value_counts()

domain
facebook.com     831159
youtube.com      725434
twitter.com      180128
tumblr.com       172095
instagram.com    106328
Name: count, dtype: int64

In [12]:
# concat with domains
domains_all = pd.concat([
    domains[["url", "unraveled_url", "domain"]],
    missing], axis=0)
len(domains_all), domains_all.url.nunique(), domains_all.domain.nunique()

(8735079, 8658030, 156579)

In [22]:
urls.url.nunique() == domains_all.url.nunique()

True

In [21]:
nan_domains = domains_all["domain"].isna().sum()
print("{} missing domains ({:1.2f}%)".format(\
        nan_domains,
        (nan_domains / len(domains_all))
        * 100))

17478 missing domains (0.20%)


In [ ]:
# save unraveled_urls 
domains_all.to_csv(join(src, "unraveled_urls_all.csv.gz"), 
                     compression="gzip", index=False)